# 13.9 · 评估生成模型 / Evaluating Generative Models (FID & IS)

> **课程定位 / Where this fits**
> 第 9 课，**Part 13 · 生成模型**(本部分收尾)。生成的图到底好不好, 怎么客观打分?
> Lesson 9, **Part 13 · Generative Models** (finale). How to objectively score how good generated images are?
>
> 评估生成模型出了名地难——没有"标准答案", 而且要同时衡量两件事: **保真度(fidelity, 生成的图像不像真的)** 和**多样性(diversity, 是否覆盖了所有类型, 还是模式坍塌只会生成几种)**。光看几张图不够客观。业界标准是两个指标: **Inception Score(IS)** 和 **Fréchet Inception Distance(FID)**。本课**从零实现 IS 和 FID**, 理解它们如何分别/同时捕捉保真度与多样性, 并用受控实验展示"图像质量下降时指标如何变化"。
> Evaluating generative models is notoriously hard — no "ground truth," and you must measure two things at once: **fidelity** (do images look real?) and **diversity** (do they cover all types, or has the model mode-collapsed to a few?). Eyeballing a few images isn't objective. The industry standards are **Inception Score (IS)** and **Fréchet Inception Distance (FID)**. We **implement IS and FID from scratch**, understand how they capture fidelity and diversity, and use controlled experiments to show how the metrics react as image quality degrades.
>
> 💼 **实战/面试视角**："FID 怎么算/越低越好 / IS 衡量什么 / FID vs IS / 这些指标的坑(需Inception/样本量敏感)" 是生成模型评估常考。
> 💼 **Practical/interview angle:** "how FID works / what IS measures / FID vs IS / pitfalls (needs Inception/sample-size sensitive)" — common.

> 📐 **符号约定 / Notation**
> - 保真度 / 多样性 —— fidelity (realism) / diversity (coverage)
> - FID —— 真实与生成的特征分布间的 Fréchet 距离(越低越好) / Fréchet distance between feature distributions (lower=better)

> 💡 **面试相关 / Interview-relevant**
> - "FID 的计算与直觉(特征分布的距离)"（出镜率 ★★★★★）
> - "Inception Score 衡量什么(质量+多样性)"（★★★★）
> - "FID 为什么比 IS 更常用"（★★★★）
> - "这些指标的局限(依赖Inception/样本量/不抓某些缺陷)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解评估的两个维度:保真度与多样性。
   Understand the two axes: fidelity and diversity.
2. **从零实现 Inception Score**, 理解它如何捕捉质量+多样性。
   Implement Inception Score from scratch; how it captures quality + diversity.
3. **从零实现 FID**, 理解特征分布距离的直觉。
   Implement FID from scratch; the feature-distribution-distance intuition.
4. 用受控实验看指标如何随质量下降变化, 并知道其局限。
   See metrics react to degraded quality in controlled experiments; know limits.

## 目录 / TOC
1. [保真度 vs 多样性 ⭐](#1)
2. [Inception Score（从零）⭐](#2)
3. [FID（从零）⭐](#3)
4. [对比、局限与小结 ⭐](#4)


<a id="1"></a>
## 1. 保真度 vs 多样性 ⭐ / Fidelity vs Diversity

一个好的生成模型要同时做到两点(面试核心)：
A good generative model must achieve two things at once (interview core):
- **保真度(fidelity)**:生成的每一张图都**清晰、真实、像那么回事**(不是模糊噪点)。
  **Fidelity:** each generated image is **clear, realistic** (not blurry noise).
- **多样性(diversity)**:生成的图**覆盖真实数据的所有类型**(各种数字都能生成), 而不是**模式坍塌**只会反复生成几种。
  **Diversity:** generations **cover all types** in the real data (all digits), not **mode-collapsed** to a few.

**为什么两者都要**:只看保真度会被"模式坍塌"骗——一个只会生成完美数字"1"的模型, 每张都很真(高保真), 但毫无多样性, 是个坏生成器。只看多样性会被噪声骗——纯随机噪声"多样"但毫无保真度。好指标要**同时惩罚这两种失败**。
**Why both:** measuring only fidelity is fooled by mode collapse — a model generating only perfect "1"s is realistic per image (high fidelity) but has no diversity, a bad generator. Measuring only diversity is fooled by noise — random noise is "diverse" but has zero fidelity. A good metric must **penalize both failures**.

IS 和 FID 都试图同时捕捉这两面。我们先训练一个 MNIST 分类器当"评判工具"(真实 FID/IS 用 ImageNet 上的 Inception 网络, 这里用自己的小分类器演示原理)。
Both IS and FID try to capture both. We first train an MNIST classifier as the "judge tool" (real FID/IS use an ImageNet Inception net; we use our own small classifier to demonstrate the principle).


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from scipy.linalg import sqrtm
sns.set_theme(style="whitegrid"); torch.manual_seed(0); np.random.seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
mnist = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(Subset(mnist, range(15000)), batch_size=128, shuffle=True)

# 训练一个 MNIST 分类器, 当作"Inception"特征提取器+质量评判 / a classifier as the "Inception" feature extractor + judge
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                  nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                  nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU())   # 64维特征(给FID用) / 64-d features
        self.head = nn.Linear(64, 10)                                              # 分类头(给IS用) / classifier head
    def forward(self, x): f = self.feat(x); return self.head(f), f
clf = Net(); opt = torch.optim.Adam(clf.parameters(), 1e-3)
for ep in range(4):
    for xb, yb in loader: opt.zero_grad(); F.cross_entropy(clf(xb)[0], yb).backward(); opt.step()
clf.eval()
acc = np.mean([ (clf(xb)[0].argmax(1)==yb).float().mean().item() for xb,yb in DataLoader(Subset(mnist,range(15000,17000)),256)])
print(f"评判用分类器训练完成, 测试准确率 ≈ {acc:.3f} (它的预测/特征用来算 IS/FID)")

# 准备几组"待评估的图像集": 真实 / 加噪 / 重度加噪 / 只有一个数字(模式坍塌) / different quality sets
real = torch.stack([mnist[i][0] for i in range(17000, 19000)])
def noisy(x, s): return torch.clamp(x + s*torch.randn_like(x), 0, 1)
sets = {
    "真实图像": real,
    "轻度加噪": noisy(real, 0.5),
    "重度加噪": noisy(real, 1.2),
    "纯噪声": torch.rand_like(real),
    "模式坍塌(全是同一张)": real[0:1].repeat(2000,1,1,1),
}
print("准备了5组图像: 真实 / 轻度加噪 / 重度加噪 / 纯噪声 / 模式坍塌; 看 IS/FID 如何区分它们")


<a id="2"></a>
## 2. Inception Score（从零）⭐ / Inception Score From Scratch

**Inception Score(IS)** 用一个分类器同时衡量保真度和多样性(面试):
**Inception Score (IS)** uses a classifier to measure both fidelity and diversity:
- **保真度**:对每张生成图, 分类器应该**很自信**(预测分布 $p(y\mid x)$ **尖锐**, 低熵)——说明它清晰像某个类别。
  **Fidelity:** for each image, the classifier should be **confident** ($p(y\mid x)$ is **peaked**, low entropy) — it clearly looks like some class.
- **多样性**:把所有生成图的预测平均起来, **边缘分布 $p(y)$ 应该接近均匀**(高熵)——说明各个类别都生成到了。
  **Diversity:** averaging predictions over all images, the **marginal $p(y)$ should be near-uniform** (high entropy) — all classes are generated.

IS 把两者合起来:$\text{IS} = \exp\big(\mathbb{E}_x[\text{KL}(p(y\mid x)\,\|\,p(y))]\big)$。当**每张图自信(尖锐)** 且**整体多样($p(y)$均匀)** 时, KL 散度大, IS 高。**越高越好**。
IS combines them: $\text{IS} = \exp\big(\mathbb{E}_x[\text{KL}(p(y\mid x)\,\|\,p(y))]\big)$. When **each image is confident (peaked)** and **the set is diverse (uniform $p(y)$)**, the KL is large and IS is high. **Higher is better.**


In [ ]:
def inception_score(images):
    with torch.no_grad():
        logits, _ = clf(images)
        py_x = F.softmax(logits, dim=1)                  # 每张图的预测分布 p(y|x) / per-image distribution
    py = py_x.mean(0, keepdim=True)                      # 边缘分布 p(y) / marginal distribution
    kl = (py_x * (torch.log(py_x + 1e-10) - torch.log(py + 1e-10))).sum(1)   # 每张图的 KL / per-image KL
    return torch.exp(kl.mean()).item()                   # exp(平均KL) / exp(mean KL)

print("Inception Score (越高越好, 同时要求'每张自信'+'整体多样'):")
for name, imgs in sets.items():
    print(f"  {name:22}: IS = {inception_score(imgs):.2f}")
print("\n真实图像IS最高(清晰+多样); 模式坍塌IS低(虽自信但不多样, p(y)不均匀); 纯噪声IS低(不自信)")
print("IS 的巧妙: 同时惩罚'不清晰'和'不多样'两种失败")


<a id="3"></a>
## 3. FID（从零）⭐ / FID From Scratch

**FID(Fréchet Inception Distance)** 是当今**最常用**的生成评估指标。思路:不看分类预测, 而是看**特征分布**——用网络(Inception/我们的分类器)把真实图和生成图都映射成特征向量, 然后**衡量这两堆特征的分布有多接近**。
**FID (Fréchet Inception Distance)** is today's **most-used** metric. Idea: not classification, but **feature distributions** — map both real and generated images to feature vectors (via Inception/our classifier), then **measure how close the two feature distributions are.**

把每堆特征近似成一个**多维高斯**(用均值 $\mu$ 和协方差 $\Sigma$ 描述), FID 就是两个高斯之间的 **Fréchet 距离**:
Approximate each feature set as a **multivariate Gaussian** ($\mu, \Sigma$); FID is the **Fréchet distance** between the two Gaussians:

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\big(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\big)$$

第一项比较**均值**(整体特征中心差多少), 第二项比较**协方差**(特征的分布形状/多样性差多少)。**FID 越低 = 生成分布越接近真实分布 = 越好**。它同时反映保真度(均值)和多样性(协方差), 这是它优于 IS 的地方。
The first term compares **means** (how far the overall feature centers are), the second compares **covariances** (how different the spread/diversity is). **Lower FID = generated distribution closer to real = better.** It reflects both fidelity (mean) and diversity (covariance) — its edge over IS.


In [ ]:
def get_features(images):
    with torch.no_grad():
        feats = []
        for i in range(0, len(images), 256):
            _, f = clf(images[i:i+256]); feats.append(f)
    return torch.cat(feats).numpy()                      # (N, 64) 特征 / features

def fid(feat_r, feat_g):
    mu_r, mu_g = feat_r.mean(0), feat_g.mean(0)          # 两组特征的均值 / means
    cov_r, cov_g = np.cov(feat_r, rowvar=False), np.cov(feat_g, rowvar=False)   # 协方差 / covariances
    diff = ((mu_r - mu_g)**2).sum()                      # 均值差(保真度) / mean difference (fidelity)
    covmean = sqrtm(cov_r @ cov_g)                       # 协方差几何平均 / matrix sqrt
    if np.iscomplexobj(covmean): covmean = covmean.real  # 数值误差可能产生小虚部 / drop tiny imaginary part
    return float(diff + np.trace(cov_r + cov_g - 2*covmean))   # FID 公式 / FID formula

feat_real = get_features(real)                           # 真实图的特征(基准) / reference features
print("FID (越低越好, =生成特征分布与真实分布的距离):")
fids = {}
for name, imgs in sets.items():
    fids[name] = fid(feat_real, get_features(imgs)); print(f"  {name:22}: FID = {fids[name]:.1f}")
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(list(fids.keys()), list(fids.values()), color=["#2a9d8f","#8ab","#e9c46a","#e76f51","#9b5"])
ax.set_ylabel("FID (越低越好)"); ax.set_title("FID: 真实≈0, 质量越差/越不像真实分布→FID越高")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()
print("\n真实图像 FID≈0(和自己比); 加噪越重 FID越高; 纯噪声FID很高; 模式坍塌FID也高(协方差差异大)")
print("FID 同时抓保真度(均值差)和多样性(协方差差) → 比IS更全面, 是当今主流指标")


<a id="4"></a>
## 4. 对比、局限与小结 ⭐ / Comparison, Limits & Summary

**FID vs IS**(面试):
**FID vs IS** (interview):
- **IS** 只用生成图(不看真实图), 且依赖分类器的类别——对**类别外**的数据(如人脸, 没有 ImageNet 类别)不适用; 也抓不到"生成图和真实图分布差异"。
  **IS** uses only generated images (not real ones) and relies on classifier classes — fails on **out-of-class** data (e.g. faces, no ImageNet class); can't capture "generated vs real distribution gap."
- **FID** 直接比较生成分布和真实分布的特征, 同时反映保真+多样, **更全面、更常用**。FID 低基本意味着"既真实又多样"。
  **FID** directly compares generated vs real feature distributions, reflecting both fidelity+diversity — **more comprehensive, more used**. Low FID basically means "realistic and diverse."

**这些指标的坑(面试)**：
**Pitfalls (interview):**
- **依赖特征提取器**:标准 FID 用 **ImageNet 上的 Inception 网络**(我们这里用自训练的小分类器演示原理, 数值不可与论文直接比)。
  **Depends on the feature extractor:** standard FID uses the **ImageNet Inception net** (we used a small self-trained classifier to show the principle; numbers aren't comparable to papers).
- **样本量敏感**:FID 用太少样本会**系统性偏高**, 要用足够多(常 1 万~5 万)且固定数量比较。
  **Sample-size sensitive:** too few samples **biases FID upward**; use enough (often 10k–50k) and a fixed count when comparing.
- **抓不到所有缺陷**:FID/IS 高分不代表没问题(可能有人类一眼可见的诡异瑕疵); 重要场景仍需**人工评估**。文生图还用 **CLIP score** 衡量图文匹配度。
  **Misses some flaws:** good FID/IS doesn't guarantee no problems (weird artifacts humans spot instantly); critical cases still need **human eval**. Text-to-image also uses **CLIP score** for image-text alignment.

```
评估两维度: 保真度(像不像真) + 多样性(覆盖所有类型, 非模式坍塌); 好指标要同时惩罚两种失败
Inception Score(IS): 用分类器; 每张自信(p(y|x)尖锐=保真)+整体多样(p(y)均匀=多样); IS=exp(E[KL(p(y|x)||p(y))]); 越高越好
FID: 比较真实vs生成的'特征分布'(各近似为高斯); FID=||μr-μg||²+Tr(Σr+Σg-2(ΣrΣg)^½); 越低越好; 同时抓保真(均值)+多样(协方差)
FID vs IS: FID直接比真实分布更全面更常用; IS只用生成图、依赖类别
坑: 依赖Inception特征提取器/样本量敏感(太少偏高)/抓不到所有瑕疵→重要场景需人工评估; 文生图用CLIP score
```

### 💡 面试速查 / Interview cheat-sheet
1. **两维度**: 保真度(像真)+多样性(覆盖); 都要, 否则被模式坍塌/噪声骗。
   Two axes: fidelity + diversity; both needed, else fooled by collapse/noise.
2. **IS**: exp(E[KL(p(y|x)||p(y))]); 每张自信+整体多样; 越高越好。
   IS: exp(E[KL]); confident per image + diverse overall; higher better.
3. **FID**: 真实vs生成特征分布(高斯)的Fréchet距离; 越低越好; 抓保真+多样。
   FID: Fréchet distance of real vs generated feature Gaussians; lower better.
4. **FID>IS**: FID直接比真实分布、更全面, 当今主流。
   FID > IS: compares to the real distribution, more comprehensive, mainstream.
5. **坑**: 依赖Inception/样本量敏感/不抓所有瑕疵; 需人工评估+CLIP score。
   Pitfalls: Inception-dependent/sample-sensitive/misses flaws; need human eval + CLIP score.

### 🎉 Part 13 完成 / Part 13 Complete
你已**从零**走完生成模型:自编码器、**VAE**、**GAN**(及 DCGAN/WGAN/cGAN)、**流模型**、**扩散模型(DDPM)**、Stable Diffusion 原理、生成评估(FID/IS)。这条线覆盖了 AIGC 时代最核心的技术——从"把数据压缩重建"到"从噪声生成以假乱真的图像", 从对抗博弈到扩散去噪。这些正是当今最热门的 AI 方向之一。
You've gone through generative models **from scratch**: autoencoders, **VAE**, **GAN** (and DCGAN/WGAN/cGAN), **normalizing flows**, **diffusion (DDPM)**, Stable Diffusion internals, and generative evaluation (FID/IS). This covers the core tech of the AIGC era — from "compress and reconstruct" to "generate realistic images from noise," from adversarial games to diffusion denoising. One of today's hottest AI areas.
